In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/isic-flagship-project"
os.chdir(PROJECT_ROOT)

In [ ]:
%%writefile src/inference/ensemble_engine.py
"""
Ensemble inference for ISIC 2024 skin lesion prediction.
"""

import torch
import timm
import numpy as np
from PIL import Image
import torchvision.transforms as T
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/isic-flagship-project")

class ISICEnsembleEngine:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.models = []
        self.transforms = []
        self.models_loaded = False

    def load_models(self):
        if self.models_loaded:
            return

        model1 = timm.create_model('convnext_small', pretrained=True, num_classes=2)
        model1 = model1.to(self.device).eval()
        self.models.append(model1)
        self.transforms.append(T.Compose([
            T.Resize(384), T.CenterCrop(384), T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]))

        model2 = timm.create_model('eva02_small_patch14_336', pretrained=True, num_classes=2)
        model2 = model2.to(self.device).eval()
        self.models.append(model2)
        self.transforms.append(T.Compose([
            T.Resize(336), T.CenterCrop(336), T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]))

        self.models_loaded = True

    def predict(self, image: Image.Image = None):
        if not self.models_loaded:
            self.load_models()

        if image is None:
            image = Image.fromarray(np.random.randint(0, 255, (384, 384, 3), dtype=np.uint8))

        probs = []
        with torch.no_grad():
            for i, model in enumerate(self.models):
                input_tensor = self.transforms[i](image).unsqueeze(0).to(self.device)
                logits = model(input_tensor)
                prob = torch.softmax(logits, dim=1)[:, 1].item()
                probs.append(prob)

        final_prob = np.mean(probs)

        return {
            "probability": final_prob,
            "prediction": "benign" if final_prob < 0.5 else "malignant"
        }


Overwriting src/inference/ensemble_engine.py


In [ ]:
%%writefile src/services/prediction_service.py
"""
Prediction service that uses the real ensemble engine.
"""

from fastapi import UploadFile
from PIL import Image
import io
from src.inference.ensemble_engine import ISICEnsembleEngine

class PredictionService:
    def __init__(self):
        self.engine = ISICEnsembleEngine()

    async def predict(self, file: UploadFile):
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        self.engine.load_models()
        result = self.engine.predict(image)

        return {
            "probability": result["probability"],
            "prediction": result["prediction"],
            "model_version": "2024-ensemble-2models"
        }

Overwriting src/services/prediction_service.py


In [ ]:
%%writefile src/api/routes.py
"""
API routes for skin lesion prediction.
"""

from fastapi import APIRouter, UploadFile, File, HTTPException
from datetime import datetime
from src.services.prediction_service import PredictionService
from src.schemas.prediction import PredictionResponse

prediction_service = PredictionService()
prediction_router = APIRouter(prefix="/api/v1", tags=["prediction"])

@prediction_router.post("/predict", response_model=PredictionResponse)
async def predict_image(file: UploadFile = File(...)):
    if not file.content_type or not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="File must be an image")

    result = await prediction_service.predict(file)

    return PredictionResponse(
        isic_id=file.filename or "uploaded_image",
        probability=result["probability"],
        prediction=result["prediction"],
        model_version=result["model_version"],
        timestamp=datetime.utcnow()
    )

Overwriting src/api/routes.py


In [ ]:
%%writefile src/api/main.py
"""
Main FastAPI application.
"""

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from src.core.config import get_settings
from src.api.routes import prediction_router

settings = get_settings()

app = FastAPI(
    title=settings.APP_NAME,
    version=settings.API_VERSION,
    description="ISIC 2024 Skin Cancer Detection"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

app.include_router(prediction_router)

@app.get("/api/v1/health")
async def health_check():
    return {
        "status": "healthy",
        "model_version": "2024-ensemble-models"
    }

@app.get("/")
async def root():
    return {"message": "ISIC 2024 Flagship API is running"}



Overwriting src/api/main.py
